# ChildWhisper — Self-Contained Training Notebook

**Everything is inline — no external .py files needed.**

| Section | Description |
|---------|-------------|
| 1 | Install & Imports |
| 2 | Configuration |
| 3 | Environment Setup (paths, auth, data verification) |
| 4 | Core Utilities (audio, text, dataset, augmentation, evaluation) |
| 5 | Whisper-small Full Fine-tune |
| 6 | Whisper-large-v3 LoRA Fine-tune |
| 7 | EDA & Visualization |
| 8 | Augmented Retraining |
| 9 | Evaluation & Error Analysis |
| 10 | Ensemble Inference & Submission |
| 11 | Summary |

---
## 1. Install & Imports

In [ ]:
import subprocess, sys
for dep in ["jiwer>=3.0", "audiomentations>=0.36", "peft>=0.12", "bitsandbytes>=0.43", "accelerate>=0.33", "librosa", "soundfile"]:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", dep])
print("Dependencies installed.")

In [ ]:
import json
import logging
import os
import random
import re
import tempfile
import time
from collections import defaultdict
from pathlib import Path
from typing import Any, Callable

import jiwer
import librosa
import numpy as np
import torch
from torch.utils.data import Dataset
from transformers import (
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments,
    WhisperForConditionalGeneration,
    WhisperProcessor,
)
from transformers.models.whisper.english_normalizer import EnglishTextNormalizer

import matplotlib
import matplotlib.pyplot as plt
matplotlib.rcParams.update({
    "figure.figsize": (12, 6),
    "figure.dpi": 100,
    "axes.grid": True,
    "grid.alpha": 0.3,
    "font.size": 11,
})

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger("childwhisper")

print(f"PyTorch: {torch.__version__}")
print(f"CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
elif hasattr(torch.backends, 'mps') and torch.backends.mps.is_available():
    print("Device: Apple Silicon (MPS)")
else:
    print("Device: CPU")

---
## 2. Configuration

Edit this cell to change training parameters. No external YAML needed.

In [ ]:
# ══════════════════════════════════════════════════════════════
# TRAINING CONFIG — All settings in one place
# ══════════════════════════════════════════════════════════════

CONFIG = {
    # --- Common ---
    "sample_rate": 16000,
    "max_duration_sec": 30.0,
    "min_duration_sec": 0.3,
    "silence_threshold_db": -40,
    "trim_top_db": 30,
    "spec_augment": {
        "apply": True,
        "mask_time_prob": 0.05,
        "mask_time_length": 10,
        "mask_feature_prob": 0.04,
        "mask_feature_length": 10,
    },
    "validation": {
        "split_ratio": 0.1,
        "stratify_by": "age_bucket",
        "split_by": "child_id",
    },
    "augmentation": {
        "realclass_min_snr_db": 5.0,
        "realclass_max_snr_db": 20.0,
        "musan_min_snr_db": 0.0,
        "musan_max_snr_db": 15.0,
    },

    # --- Data Subset (to reduce training time) ---
    # Set to 1.0 to use all data, 0.3 for ~68k samples (~10-12 hrs on T4x2)
    # Sampling is stratified by age_bucket at the child_id level (no leakage)
    "data_subset_fraction": 0.3,
    "data_subset_seed": 42,

    # --- Whisper-small ---
    "whisper_small": {
        "model_name": "openai/whisper-small",
        "learning_rate": 1e-5,
        "warmup_steps": 500,
        "num_train_epochs": 3,
        "per_device_train_batch_size": 8,
        "per_device_eval_batch_size": 8,
        "gradient_accumulation_steps": 4,
        "fp16": True,
        "gradient_checkpointing": True,
        "eval_steps": 500,
        "save_steps": 500,
        "save_total_limit": 3,
        "generation_max_length": 225,
        "dataloader_num_workers": 4,
        "hub_model_id": "nishantgaurav23/pasketti-whisper-small",
        "hub_private_repo": True,
    },

    # --- Whisper-large-v3 LoRA ---
    "whisper_large_v3": {
        "model_name": "openai/whisper-large-v3",
        "learning_rate": 1e-4,
        "warmup_steps": 500,
        "num_train_epochs": 3,
        "per_device_train_batch_size": 2,
        "per_device_eval_batch_size": 4,
        "gradient_accumulation_steps": 8,
        "fp16": True,
        "gradient_checkpointing": True,
        "load_in_8bit": True,
        "eval_steps": 500,
        "save_steps": 500,
        "save_total_limit": 3,
        "generation_max_length": 225,
        "dataloader_num_workers": 4,
        "hub_model_id": "nishantgaurav23/pasketti-whisper-lora",
        "hub_private_repo": True,
        "lora": {
            "r": 32,
            "alpha": 64,
            "target_modules": ["q_proj", "v_proj"],
            "dropout": 0.05,
            "bias": "none",
            "task_type": "SEQ_2_SEQ_LM",
        },
    },
}

# --- What to run ---
RUN_SMALL_TRAIN = True
RUN_LORA_TRAIN = True
RUN_AUGMENTED = False
RUN_EVALUATION = True
RUN_ENSEMBLE = True
PUSH_TO_HUB = True

# HuggingFace Hub model IDs
SMALL_HUB_ID = "nishantgaurav23/pasketti-whisper-small"
LORA_HUB_ID = "nishantgaurav23/pasketti-whisper-lora"

print("Config loaded.")
frac = CONFIG["data_subset_fraction"]
if frac < 1.0:
    print(f"  Data subset: {frac*100:.0f}% (stratified by age_bucket, grouped by child_id)")
else:
    print("  Using full dataset (no subsetting)")

---
## 3. Environment Setup

In [ ]:
# ══════════════════════════════════════════════════════════════
# ENVIRONMENT DETECTION & PATHS
# ══════════════════════════════════════════════════════════════

def is_kaggle():
    return bool(os.environ.get("KAGGLE_KERNEL_RUN_TYPE")) or Path("/kaggle/working").exists()

ON_KAGGLE = is_kaggle()
print(f"Running on Kaggle: {ON_KAGGLE}")

if ON_KAGGLE:
    # Find the dataset base path (Kaggle uses different path layouts)
    DATASET_BASE = None
    for candidate in [
        Path("/kaggle/input/pasketti-audio"),
        Path("/kaggle/input/datasets/nishantgaurav23/pasketti-audio"),
    ]:
        if candidate.exists():
            DATASET_BASE = candidate
            break
    if DATASET_BASE is None:
        raise FileNotFoundError(
            "Cannot find pasketti-audio dataset. "
            "Please attach it to your Kaggle notebook."
        )
    print(f"Dataset base: {DATASET_BASE}")

    # --- Unify split audio into a single directory ---
    UNIFIED_DIR = Path("/kaggle/working/unified_audio")
    audio_out = UNIFIED_DIR / "audio"
    audio_out.mkdir(parents=True, exist_ok=True)

    count = 0
    # Symlink from audio_part_*/audio/
    for part_dir in sorted(DATASET_BASE.glob("audio_part_*/audio")):
        for f in part_dir.iterdir():
            link = audio_out / f.name
            if not link.exists():
                link.symlink_to(f)
                count += 1
    # Also check audio/audio/ (non-split layout)
    flat_dir = DATASET_BASE / "audio" / "audio"
    if flat_dir.is_dir():
        for f in flat_dir.iterdir():
            link = audio_out / f.name
            if not link.exists():
                link.symlink_to(f)
                count += 1
    print(f"Unified {count} audio files into {audio_out}")

    AUDIO_DIR = UNIFIED_DIR

    # --- Find metadata ---
    METADATA_PATH = DATASET_BASE / "train_word_transcripts.jsonl"
    if not METADATA_PATH.exists():
        for c in DATASET_BASE.glob("**/train_word_transcripts.jsonl"):
            METADATA_PATH = c
            break

    SMALL_OUTPUT_DIR = Path("/kaggle/working/checkpoints/whisper-small")
    LORA_OUTPUT_DIR = Path("/kaggle/working/checkpoints/whisper-large-v3-lora")

    # --- Noise paths ---
    NOISE_DIR = None
    noise_parts = sorted(DATASET_BASE.glob("noise_part_*/audio"))
    if noise_parts:
        noise_out = Path("/kaggle/working/unified_noise")
        noise_out.mkdir(parents=True, exist_ok=True)
        for part_dir in noise_parts:
            for f in part_dir.iterdir():
                link = noise_out / f.name
                if not link.exists():
                    link.symlink_to(f)
        NOISE_DIR = noise_out
        print(f"Unified noise files into {noise_out}")

    MUSAN_DIR = DATASET_BASE / "musan" / "musan"
    if not MUSAN_DIR.is_dir():
        MUSAN_DIR = None

else:
    # Local development
    LOCAL_DATA_DIR = Path("data")
    AUDIO_DIR = LOCAL_DATA_DIR / "audio_sample"
    METADATA_PATH = LOCAL_DATA_DIR / "train_word_transcripts.jsonl"
    SMALL_OUTPUT_DIR = Path("./checkpoints/whisper-small")
    LORA_OUTPUT_DIR = Path("./checkpoints/whisper-large-v3-lora")
    NOISE_DIR = Path("data/musan_noise") if Path("data/musan_noise").exists() else None
    MUSAN_DIR = Path("data/musan") if Path("data/musan").exists() else None

print(f"Audio dir:     {AUDIO_DIR}")
print(f"Metadata:      {METADATA_PATH}")
print(f"Small output:  {SMALL_OUTPUT_DIR}")
print(f"LoRA output:   {LORA_OUTPUT_DIR}")
print(f"Noise dir:     {NOISE_DIR}")
print(f"MUSAN dir:     {MUSAN_DIR}")

In [ ]:
# ══════════════════════════════════════════════════════════════
# HuggingFace Hub Authentication
# ══════════════════════════════════════════════════════════════

# Load Kaggle secrets into env if available
try:
    from kaggle_secrets import UserSecretsClient
    _token = UserSecretsClient().get_secret("HF_TOKEN")
    if _token:
        os.environ["HF_TOKEN"] = _token
        os.environ["HUGGING_FACE_HUB_TOKEN"] = _token
except (ImportError, Exception):
    pass

_hf_token = os.environ.get("HF_TOKEN")
if _hf_token:
    os.environ["HUGGING_FACE_HUB_TOKEN"] = _hf_token
    try:
        from huggingface_hub import login
        login(token=_hf_token, add_to_git_credential=False, new_session=False)
        print("HF Hub authenticated successfully")
    except Exception as e:
        print(f"HF login failed (token still set via env): {e}")
else:
    print("HF_TOKEN not set — checkpoints will be saved locally only")
    PUSH_TO_HUB = False

In [ ]:
# ══════════════════════════════════════════════════════════════
# DATA VERIFICATION
# ══════════════════════════════════════════════════════════════

def verify_data(audio_dir, metadata_path):
    meta_path = Path(metadata_path)
    if not meta_path.exists():
        raise FileNotFoundError(f"Metadata not found: {metadata_path}")
    text = meta_path.read_text().strip()
    entries = [json.loads(line) for line in text.split("\n")] if text else []
    audio_path = Path(audio_dir)
    found = sum(1 for e in entries if (audio_path / e["audio_path"]).exists())
    missing = len(entries) - found
    durations = [e["audio_duration_sec"] for e in entries if "audio_duration_sec" in e]
    print(f"Utterances:    {len(entries)}")
    print(f"Audio found:   {found}")
    print(f"Audio missing: {missing}")
    if durations:
        print(f"Duration: min={min(durations):.2f}s, max={max(durations):.2f}s, "
              f"mean={sum(durations)/len(durations):.2f}s, "
              f"total={sum(durations)/3600:.1f}h")
    if missing > 0:
        print(f"WARNING: {missing/len(entries)*100:.1f}% audio files missing!")
    return entries

all_entries = verify_data(AUDIO_DIR, METADATA_PATH)

---
## 4. Core Utilities

All functions that were in `src/*.py` are defined inline here.

In [ ]:
# ══════════════════════════════════════════════════════════════
# AUDIO PREPROCESSING (was src/preprocess.py)
# ══════════════════════════════════════════════════════════════

def load_audio(audio_path, target_sr=16000):
    """Load audio file and resample to 16kHz mono."""
    audio, sr = librosa.load(audio_path, sr=target_sr, mono=True)
    return audio, target_sr

def trim_silence(audio, top_db=30):
    """Trim leading/trailing silence."""
    trimmed, _ = librosa.effects.trim(audio, top_db=top_db)
    return trimmed

def is_silence(audio, threshold_db=-40.0):
    """Return True if audio RMS energy is below threshold."""
    rms = np.sqrt(np.mean(audio**2))
    db = 20 * np.log10(rms + 1e-10)
    return bool(db < threshold_db)

def get_duration(audio, sr=16000):
    return len(audio) / sr

def is_valid_duration(duration, min_dur=0.3, max_dur=30.0):
    return min_dur <= duration <= max_dur

def load_metadata(jsonl_path):
    """Load utterance metadata from JSONL file."""
    path = Path(jsonl_path)
    text = path.read_text().strip()
    if not text:
        return []
    return [json.loads(line) for line in text.split("\n")]

print("Audio preprocessing functions loaded.")

In [ ]:
# ══════════════════════════════════════════════════════════════
# TEXT NORMALIZATION & POST-PROCESSING (was src/utils.py)
# ══════════════════════════════════════════════════════════════

_normalizer = EnglishTextNormalizer({})

_ARTIFACT_PATTERN = re.compile(
    r"\u266a"               # music symbol
    r"|\[[^\]]*\]"          # [anything in brackets]
    r"|\([^)]*\)"           # (anything in parens)
    r"|\.{2,}"              # ellipsis
)

def normalize_text(text):
    """Normalize text using Whisper's EnglishTextNormalizer."""
    if text is None or not text.strip():
        return ""
    return _normalizer(text)

def _collapse_repeated_words(text):
    """Collapse 3+ consecutive identical words to single occurrence."""
    words = text.lower().split()
    if len(words) <= 2:
        return " ".join(words)
    max_phrase_len = len(words) // 3
    for phrase_len in range(max_phrase_len, 0, -1):
        i = 0
        result = []
        while i < len(words):
            phrase = words[i : i + phrase_len]
            if i + phrase_len <= len(words):
                count = 1
                j = i + phrase_len
                while j + phrase_len <= len(words) and words[j : j + phrase_len] == phrase:
                    count += 1
                    j += phrase_len
                if count >= 3:
                    result.extend(phrase)
                    i = j
                    continue
            result.append(words[i])
            i += 1
        words = result
    return " ".join(words)

def postprocess_text(text):
    """Remove artifacts, collapse repeats, clean whitespace."""
    if text is None or not text.strip():
        return ""
    text = _ARTIFACT_PATTERN.sub("", text)
    text = re.sub(r"\s+", " ", text).strip()
    text = _collapse_repeated_words(text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

def normalize_and_postprocess(text):
    """Normalize + postprocess for inference output."""
    normalized = normalize_text(text)
    if not normalized:
        return ""
    return postprocess_text(normalized)

print("Text normalization functions loaded.")

In [ ]:
# ══════════════════════════════════════════════════════════════
# DATASET & DATA COLLATOR (was src/dataset.py)
# ══════════════════════════════════════════════════════════════

class WhisperDataset(Dataset):
    """PyTorch Dataset: loads audio, extracts Whisper features, tokenizes transcripts."""

    def __init__(self, metadata_path, audio_dir, model_name="openai/whisper-small",
                 augment_fn=None, min_duration=0.3, max_duration=30.0,
                 language="en", task="transcribe"):
        self.audio_dir = Path(audio_dir)
        self.augment_fn = augment_fn
        self.min_duration = min_duration
        self.max_duration = max_duration
        self.processor = WhisperProcessor.from_pretrained(model_name)
        self.processor.tokenizer.set_prefix_tokens(language=language, task=task)
        raw_metadata = load_metadata(metadata_path)
        self.entries = self._filter_entries(raw_metadata)

    def _filter_entries(self, metadata):
        valid = []
        for entry in metadata:
            audio_path = self.audio_dir / entry["audio_path"]
            if not audio_path.exists():
                continue
            dur = entry.get("audio_duration_sec")
            if dur is not None and not is_valid_duration(dur, self.min_duration, self.max_duration):
                continue
            valid.append(entry)
        return valid

    def __len__(self):
        return len(self.entries)

    def __getitem__(self, idx):
        entry = self.entries[idx]
        audio_path = self.audio_dir / entry["audio_path"]
        audio, sr = load_audio(audio_path)
        audio = trim_silence(audio)
        transcript = "" if is_silence(audio) else normalize_text(entry.get("orthographic_text", ""))
        if self.augment_fn is not None:
            audio = self.augment_fn(audio, sample_rate=sr)
        features = self.processor.feature_extractor(audio, sampling_rate=sr, return_tensors="np")
        input_features = features.input_features[0]
        labels = self.processor.tokenizer(transcript)["input_ids"]
        return {"input_features": input_features, "labels": labels}


class WhisperDataCollator:
    """Pads input_features and labels for Whisper training."""

    def __init__(self, pad_token_id=0):
        self.pad_token_id = pad_token_id

    def __call__(self, features):
        input_features = torch.tensor(
            np.array([f["input_features"] for f in features]), dtype=torch.float32
        )
        label_lists = [f["labels"] for f in features]
        max_len = max(len(lab) for lab in label_lists)
        padded = [list(lab) + [-100] * (max_len - len(lab)) for lab in label_lists]
        labels = torch.tensor(padded, dtype=torch.long)
        return {"input_features": input_features, "labels": labels}


def create_train_val_split(metadata, val_ratio=0.1, split_by="child_id",
                           stratify_by="age_bucket", seed=42):
    """Split by child_id with age_bucket stratification (no speaker leakage)."""
    rng = random.Random(seed)
    bucket_to_children = defaultdict(set)
    child_to_entries = defaultdict(list)
    for entry in metadata:
        child = entry[split_by]
        bucket = entry.get(stratify_by, "unknown")
        bucket_to_children[bucket].add(child)
        child_to_entries[child].append(entry)
    val_children = set()
    for bucket, children in bucket_to_children.items():
        children_list = sorted(children)
        rng.shuffle(children_list)
        n_val = max(1, round(len(children_list) * val_ratio))
        val_children.update(children_list[:n_val])
    train = [e for e in metadata if e[split_by] not in val_children]
    val = [e for e in metadata if e[split_by] in val_children]
    return train, val

print("Dataset classes loaded.")

In [ ]:
# ══════════════════════════════════════════════════════════════
# NOISE AUGMENTATION (was src/augment.py)
# ══════════════════════════════════════════════════════════════

def create_augmentation(noise_dir, realclass_dir, sample_rate=16000,
                        realclass_min_snr=5.0, realclass_max_snr=20.0,
                        musan_min_snr=0.0, musan_max_snr=15.0):
    """50% RealClass + 20% MUSAN + 30% clean, with Gain variation."""
    from audiomentations import AddBackgroundNoise, Compose, Gain, OneOf
    transform = Compose([
        OneOf([
            AddBackgroundNoise(sounds_path=str(realclass_dir),
                               min_snr_db=realclass_min_snr, max_snr_db=realclass_max_snr, p=1.0),
            AddBackgroundNoise(sounds_path=str(noise_dir),
                               min_snr_db=musan_min_snr, max_snr_db=musan_max_snr, p=1.0),
        ], weights=[5, 2], p=0.7),
        Gain(min_gain_db=-6, max_gain_db=6, p=0.3),
    ])
    def augment_fn(audio, sample_rate=sample_rate):
        augmented = transform(samples=audio, sample_rate=sample_rate)
        return np.asarray(augmented, dtype=audio.dtype).ravel()
    return augment_fn

print("Augmentation functions loaded.")

In [ ]:
# ══════════════════════════════════════════════════════════════
# EVALUATION — WER (was src/evaluate.py)
# ══════════════════════════════════════════════════════════════

def compute_wer(references, hypotheses):
    """Compute WER with Whisper normalizer. Skips empty refs."""
    filtered_refs, filtered_hyps = [], []
    for ref, hyp in zip(references, hypotheses):
        norm_ref = normalize_text(ref)
        if not norm_ref.strip():
            continue
        filtered_refs.append(norm_ref)
        filtered_hyps.append(normalize_text(hyp))
    if not filtered_refs:
        return 0.0
    return jiwer.wer(filtered_refs, filtered_hyps)

def compute_per_age_wer(references, hypotheses, age_buckets):
    """WER per age bucket."""
    bucket_refs = defaultdict(list)
    bucket_hyps = defaultdict(list)
    for ref, hyp, age in zip(references, hypotheses, age_buckets):
        bucket_refs[age].append(ref)
        bucket_hyps[age].append(hyp)
    return {b: compute_wer(bucket_refs[b], bucket_hyps[b]) for b in sorted(bucket_refs)}

print("Evaluation functions loaded.")

In [ ]:
# ══════════════════════════════════════════════════════════════
# SHARED TRAINING HELPERS
# ══════════════════════════════════════════════════════════════

def stratified_subset(metadata, fraction, seed=42):
    """Select a fraction of data, stratified by age_bucket at child_id level.

    Ensures:
    - No child is split across subset/remainder (no leakage)
    - Age bucket distribution is preserved
    - Rare buckets always keep at least 1 child
    """
    if fraction >= 1.0:
        return metadata

    rng = random.Random(seed)

    # Group samples by child_id
    child_to_samples = defaultdict(list)
    for entry in metadata:
        child_to_samples[entry.get("child_id", "unknown")].append(entry)

    # Group children by age_bucket (use first sample's bucket)
    bucket_to_children = defaultdict(list)
    for child_id, samples in child_to_samples.items():
        bucket = samples[0].get("age_bucket", "unknown")
        bucket_to_children[bucket].append(child_id)

    # Sample fraction of children per bucket
    selected = []
    print(f"\nStratified sampling ({fraction*100:.0f}% of data):")
    print(f"{'Bucket':<15} {'Children':<12} {'Selected':<12} {'Samples'}")
    print("-" * 55)

    for bucket in sorted(bucket_to_children):
        children = bucket_to_children[bucket]
        rng.shuffle(children)
        n_select = max(1, round(len(children) * fraction))
        for child_id in children[:n_select]:
            selected.extend(child_to_samples[child_id])
        n_samples = sum(len(child_to_samples[c]) for c in children[:n_select])
        print(f"{bucket:<15} {len(children):<12} {n_select:<12} {n_samples}")

    print(f"\nSubset: {len(selected)} / {len(metadata)} samples "
          f"({len(selected)/len(metadata)*100:.1f}%)")
    return selected


def build_datasets(model_config, metadata_path, audio_dir, augment_fn=None):
    """Build train/val WhisperDatasets with child_id split and optional subsetting."""
    metadata = load_metadata(metadata_path)

    # Apply stratified subset BEFORE train/val split
    frac = CONFIG.get("data_subset_fraction", 1.0)
    if frac < 1.0:
        metadata = stratified_subset(
            metadata, frac, seed=CONFIG.get("data_subset_seed", 42)
        )

    val_cfg = CONFIG["validation"]
    train_meta, val_meta = create_train_val_split(
        metadata,
        val_ratio=val_cfg["split_ratio"],
        split_by=val_cfg["split_by"],
        stratify_by=val_cfg["stratify_by"],
    )
    print(f"Train: {len(train_meta)} utterances, Val: {len(val_meta)} utterances")

    # Write split metadata to temp files
    train_path = Path(tempfile.mktemp(suffix=".jsonl"))
    val_path = Path(tempfile.mktemp(suffix=".jsonl"))
    train_path.write_text("\n".join(json.dumps(e) for e in train_meta))
    val_path.write_text("\n".join(json.dumps(e) for e in val_meta))

    model_name = model_config["model_name"]
    min_dur = CONFIG["min_duration_sec"]
    max_dur = CONFIG["max_duration_sec"]

    train_ds = WhisperDataset(str(train_path), audio_dir, model_name=model_name,
                              augment_fn=augment_fn, min_duration=min_dur, max_duration=max_dur)
    val_ds = WhisperDataset(str(val_path), audio_dir, model_name=model_name,
                            augment_fn=None, min_duration=min_dur, max_duration=max_dur)
    return train_ds, val_ds


def make_compute_metrics(tokenizer):
    """Create compute_metrics function for Seq2SeqTrainer."""
    def compute_metrics(pred):
        pred_ids = pred.predictions
        label_ids = pred.label_ids
        pad_id = getattr(tokenizer, "pad_token_id", 0) or 0
        label_ids = np.where(label_ids == -100, pad_id, label_ids)
        pred_str = tokenizer.batch_decode(pred_ids, skip_special_tokens=True)
        label_str = tokenizer.batch_decode(label_ids, skip_special_tokens=True)
        wer = compute_wer(label_str, pred_str)
        return {"wer": wer}
    return compute_metrics


def make_training_args(model_config, output_dir, push_to_hub=True, dry_run=False):
    """Build Seq2SeqTrainingArguments from config."""
    kwargs = {
        "output_dir": str(output_dir),
        "per_device_train_batch_size": model_config["per_device_train_batch_size"],
        "per_device_eval_batch_size": model_config["per_device_eval_batch_size"],
        "gradient_accumulation_steps": model_config["gradient_accumulation_steps"],
        "learning_rate": model_config["learning_rate"],
        "warmup_steps": model_config["warmup_steps"],
        "num_train_epochs": model_config["num_train_epochs"],
        "fp16": model_config.get("fp16", True),
        "gradient_checkpointing": model_config.get("gradient_checkpointing", True),
        "eval_strategy": "steps",
        "eval_steps": model_config["eval_steps"],
        "save_steps": model_config["save_steps"],
        "save_total_limit": model_config["save_total_limit"],
        "load_best_model_at_end": True,
        "metric_for_best_model": "wer",
        "greater_is_better": False,
        "predict_with_generate": True,
        "generation_max_length": model_config["generation_max_length"],
        "logging_steps": 50,
        "dataloader_num_workers": model_config.get("dataloader_num_workers", 4),
        "push_to_hub": push_to_hub,
        "remove_unused_columns": False,
    }
    if push_to_hub:
        kwargs["hub_model_id"] = model_config.get("hub_model_id")
        kwargs["hub_private_repo"] = model_config.get("hub_private_repo", True)
    if dry_run:
        kwargs.update({"max_steps": 1, "eval_steps": 1, "save_steps": 1,
                       "logging_steps": 1, "push_to_hub": False, "fp16": False,
                       "gradient_checkpointing": False, "dataloader_num_workers": 0})
    return Seq2SeqTrainingArguments(**kwargs)

print("Training helpers loaded.")

---
## 5. Whisper-small Full Fine-tune

In [ ]:
if RUN_SMALL_TRAIN:
    small_cfg = {**CONFIG, **CONFIG["whisper_small"]}
    n_gpu = max(1, torch.cuda.device_count()) if torch.cuda.is_available() else 1
    eff = small_cfg['per_device_train_batch_size'] * small_cfg['gradient_accumulation_steps'] * n_gpu
    frac = CONFIG.get("data_subset_fraction", 1.0)

    print("=== Whisper-small Training Config ===")
    print(f"Model:            {small_cfg['model_name']}")
    print(f"Learning rate:    {small_cfg['learning_rate']}")
    print(f"Epochs:           {small_cfg['num_train_epochs']}")
    print(f"Batch/GPU:        {small_cfg['per_device_train_batch_size']}")
    print(f"Grad accum:       {small_cfg['gradient_accumulation_steps']}")
    print(f"GPUs:             {n_gpu}")
    print(f"Effective batch:  {eff}")
    print(f"Num workers:      {small_cfg.get('dataloader_num_workers', 2)}")
    if frac < 1.0:
        print(f"Data subset:      {frac*100:.0f}%")
else:
    print("Skipping Whisper-small training (RUN_SMALL_TRAIN=False)")

In [ ]:
if RUN_SMALL_TRAIN:
    # Load model
    model_name = small_cfg["model_name"]
    small_model = WhisperForConditionalGeneration.from_pretrained(
        model_name, torch_dtype=torch.float32
    )
    small_processor = WhisperProcessor.from_pretrained(model_name)

    # Gradient checkpointing
    if small_cfg.get("gradient_checkpointing", True):
        small_model.gradient_checkpointing_enable()

    # SpecAugment
    sa = CONFIG["spec_augment"]
    if sa["apply"]:
        small_model.config.apply_spec_augment = True
        small_model.config.mask_time_prob = sa["mask_time_prob"]
        small_model.config.mask_time_length = sa["mask_time_length"]
        small_model.config.mask_feature_prob = sa["mask_feature_prob"]
        small_model.config.mask_feature_length = sa["mask_feature_length"]

    small_model.config.forced_decoder_ids = None
    small_model.generation_config.forced_decoder_ids = None
    small_model.generation_config.suppress_tokens = []

    print(f"Model loaded: {model_name}")
    print(f"Parameters: {small_model.num_parameters():,}")

In [ ]:
if RUN_SMALL_TRAIN:
    # Build datasets
    small_train_ds, small_val_ds = build_datasets(
        CONFIG["whisper_small"], str(METADATA_PATH), str(AUDIO_DIR)
    )
    print(f"Train samples: {len(small_train_ds)}, Val samples: {len(small_val_ds)}")

In [ ]:
if RUN_SMALL_TRAIN:
    # Train
    training_args = make_training_args(CONFIG["whisper_small"], SMALL_OUTPUT_DIR,
                                       push_to_hub=PUSH_TO_HUB)
    collator = WhisperDataCollator(pad_token_id=small_processor.tokenizer.pad_token_id)
    compute_metrics_fn = make_compute_metrics(small_processor.tokenizer)

    trainer = Seq2SeqTrainer(
        model=small_model,
        args=training_args,
        train_dataset=small_train_ds,
        eval_dataset=small_val_ds,
        data_collator=collator,
        compute_metrics=compute_metrics_fn,
        processing_class=small_processor.feature_extractor,
    )

    print("Starting Whisper-small training...")
    t0 = time.time()
    trainer.train()
    elapsed = time.time() - t0
    print(f"Training completed in {elapsed/3600:.1f} hours")

    # Evaluate
    metrics = trainer.evaluate()
    print(f"Validation WER: {metrics.get('eval_wer', -1):.4f}")

    # Save
    trainer.save_model()
    small_processor.save_pretrained(str(SMALL_OUTPUT_DIR))
    print(f"Model saved to {SMALL_OUTPUT_DIR}")

---
## 6. Whisper-large-v3 LoRA Fine-tune

In [ ]:
if RUN_LORA_TRAIN:
    lora_cfg = {**CONFIG, **CONFIG["whisper_large_v3"]}
    n_gpu = max(1, torch.cuda.device_count()) if torch.cuda.is_available() else 1
    eff = lora_cfg['per_device_train_batch_size'] * lora_cfg['gradient_accumulation_steps'] * n_gpu
    frac = CONFIG.get("data_subset_fraction", 1.0)

    print("=== Whisper-large-v3 LoRA Training Config ===")
    print(f"Model:            {lora_cfg['model_name']}")
    print(f"Learning rate:    {lora_cfg['learning_rate']}")
    print(f"Epochs:           {lora_cfg['num_train_epochs']}")
    print(f"Batch/GPU:        {lora_cfg['per_device_train_batch_size']}")
    print(f"Grad accum:       {lora_cfg['gradient_accumulation_steps']}")
    print(f"GPUs:             {n_gpu}")
    print(f"Effective batch:  {eff}")
    print(f"INT8:             {lora_cfg.get('load_in_8bit', False)}")
    print(f"Num workers:      {lora_cfg.get('dataloader_num_workers', 2)}")
    if frac < 1.0:
        print(f"Data subset:      {frac*100:.0f}%")
    lora_params = lora_cfg["lora"]
    print(f"LoRA r={lora_params['r']}, alpha={lora_params['alpha']}, "
          f"targets={lora_params['target_modules']}")

    # Check GPU memory
    if torch.cuda.is_available():
        for gpu_i in range(torch.cuda.device_count()):
            props = torch.cuda.get_device_properties(gpu_i)
            gpu_gb = props.total_memory / (1024**3)
            print(f"GPU {gpu_i}: {props.name} ({gpu_gb:.1f} GB)")
            if gpu_gb < 14:
                print(f"  WARNING: <14GB VRAM — LoRA training may OOM on GPU {gpu_i}")
    else:
        print("WARNING: No GPU detected")
else:
    print("Skipping LoRA training (RUN_LORA_TRAIN=False)")

In [ ]:
if RUN_LORA_TRAIN:
    from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

    model_name = lora_cfg["model_name"]
    use_8bit = lora_cfg.get("load_in_8bit", False)

    load_kwargs = {}
    if use_8bit:
        load_kwargs["load_in_8bit"] = True
        load_kwargs["device_map"] = "auto"
    else:
        load_kwargs["torch_dtype"] = torch.float16 if lora_cfg.get("fp16", True) else torch.float32

    lora_model = WhisperForConditionalGeneration.from_pretrained(model_name, **load_kwargs)
    lora_processor = WhisperProcessor.from_pretrained(model_name)

    if use_8bit:
        lora_model = prepare_model_for_kbit_training(lora_model)

    if lora_cfg.get("gradient_checkpointing", True):
        lora_model.gradient_checkpointing_enable()

    # SpecAugment
    sa = CONFIG["spec_augment"]
    if sa["apply"]:
        lora_model.config.apply_spec_augment = True
        lora_model.config.mask_time_prob = sa["mask_time_prob"]
        lora_model.config.mask_time_length = sa["mask_time_length"]
        lora_model.config.mask_feature_prob = sa["mask_feature_prob"]
        lora_model.config.mask_feature_length = sa["mask_feature_length"]

    lora_model.config.forced_decoder_ids = None
    lora_model.config.suppress_tokens = []
    lora_model.generation_config.forced_decoder_ids = None

    # Apply LoRA
    lp = lora_cfg["lora"]
    peft_config = LoraConfig(
        r=lp["r"], lora_alpha=lp["alpha"], target_modules=lp["target_modules"],
        lora_dropout=lp["dropout"], bias=lp["bias"], task_type=lp["task_type"],
    )
    lora_model = get_peft_model(lora_model, peft_config)
    lora_model.print_trainable_parameters()
    print(f"Model loaded with LoRA: {model_name}")

In [ ]:
if RUN_LORA_TRAIN:
    lora_train_ds, lora_val_ds = build_datasets(
        CONFIG["whisper_large_v3"], str(METADATA_PATH), str(AUDIO_DIR)
    )
    print(f"Train samples: {len(lora_train_ds)}, Val samples: {len(lora_val_ds)}")

In [ ]:
if RUN_LORA_TRAIN:
    training_args = make_training_args(CONFIG["whisper_large_v3"], LORA_OUTPUT_DIR,
                                       push_to_hub=PUSH_TO_HUB)
    collator = WhisperDataCollator(pad_token_id=lora_processor.tokenizer.pad_token_id)
    compute_metrics_fn = make_compute_metrics(lora_processor.tokenizer)

    trainer = Seq2SeqTrainer(
        model=lora_model,
        args=training_args,
        train_dataset=lora_train_ds,
        eval_dataset=lora_val_ds,
        data_collator=collator,
        compute_metrics=compute_metrics_fn,
        processing_class=lora_processor.feature_extractor,
    )

    print("Starting LoRA training...")
    t0 = time.time()
    trainer.train()
    elapsed = time.time() - t0
    print(f"Training completed in {elapsed/3600:.1f} hours")

    metrics = trainer.evaluate()
    print(f"Validation WER: {metrics.get('eval_wer', -1):.4f}")

    # Save adapter only
    lora_model.save_pretrained(str(LORA_OUTPUT_DIR))
    lora_processor.save_pretrained(str(LORA_OUTPUT_DIR))
    print(f"LoRA adapter saved to {LORA_OUTPUT_DIR}")

---
## 7. EDA & Visualization

Explore the dataset: duration distributions, age buckets, speaker counts, transcript lengths, and audio samples.

In [ ]:
# ══════════════════════════════════════════════════════════════
# 7a. Duration Distribution — Histogram + Box Plot by Age Bucket
# ══════════════════════════════════════════════════════════════

metadata = load_metadata(str(METADATA_PATH))
durations = [e.get("audio_duration_sec", 0) for e in metadata]
age_buckets = [e.get("age_bucket", "unknown") for e in metadata]

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Histogram
axes[0].hist(durations, bins=60, color="steelblue", edgecolor="white", alpha=0.85)
axes[0].set_xlabel("Duration (seconds)")
axes[0].set_ylabel("Count")
axes[0].set_title("Audio Duration Distribution")
axes[0].axvline(np.median(durations), color="red", linestyle="--", label=f"Median: {np.median(durations):.2f}s")
axes[0].legend()

# Box plot by age bucket
bucket_names = sorted(set(age_buckets))
bucket_durations = [[d for d, a in zip(durations, age_buckets) if a == b] for b in bucket_names]
bp = axes[1].boxplot(bucket_durations, labels=bucket_names, patch_artist=True)
colors = plt.cm.Set2(np.linspace(0, 1, len(bucket_names)))
for patch, color in zip(bp["boxes"], colors):
    patch.set_facecolor(color)
axes[1].set_xlabel("Age Bucket")
axes[1].set_ylabel("Duration (seconds)")
axes[1].set_title("Duration by Age Bucket")

plt.tight_layout()
plt.savefig("eda_duration_distribution.png", dpi=150, bbox_inches="tight")
plt.show()
print(f"Total utterances: {len(metadata)}, Duration range: {min(durations):.2f}s - {max(durations):.2f}s")

In [ ]:
# ══════════════════════════════════════════════════════════════
# 7b. Age Bucket Distribution, Speaker Count, Transcript Length
# ══════════════════════════════════════════════════════════════

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Age bucket distribution
bucket_counts = defaultdict(int)
for e in metadata:
    bucket_counts[e.get("age_bucket", "unknown")] += 1
sorted_buckets = sorted(bucket_counts.keys())
counts = [bucket_counts[b] for b in sorted_buckets]
bars = axes[0].bar(sorted_buckets, counts, color="coral", edgecolor="white")
axes[0].set_xlabel("Age Bucket")
axes[0].set_ylabel("Utterance Count")
axes[0].set_title("Utterances per Age Bucket")
for bar, count in zip(bars, counts):
    axes[0].text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 5,
                 str(count), ha="center", va="bottom", fontsize=9)

# Unique speakers per age bucket
bucket_speakers = defaultdict(set)
for e in metadata:
    bucket_speakers[e.get("age_bucket", "unknown")].add(e.get("child_id", ""))
speaker_counts = [len(bucket_speakers[b]) for b in sorted_buckets]
bars2 = axes[1].bar(sorted_buckets, speaker_counts, color="mediumpurple", edgecolor="white")
axes[1].set_xlabel("Age Bucket")
axes[1].set_ylabel("Unique Speakers")
axes[1].set_title("Speakers per Age Bucket")
for bar, count in zip(bars2, speaker_counts):
    axes[1].text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.5,
                 str(count), ha="center", va="bottom", fontsize=9)

# Transcript length distribution (word count)
transcript_lengths = [len(e.get("orthographic_text", "").split()) for e in metadata]
axes[2].hist(transcript_lengths, bins=30, color="seagreen", edgecolor="white", alpha=0.85)
axes[2].set_xlabel("Transcript Length (words)")
axes[2].set_ylabel("Count")
axes[2].set_title("Transcript Length Distribution")
axes[2].axvline(np.median(transcript_lengths), color="red", linestyle="--",
                label=f"Median: {np.median(transcript_lengths):.0f} words")
axes[2].legend()

plt.tight_layout()
plt.savefig("eda_age_speaker_transcript.png", dpi=150, bbox_inches="tight")
plt.show()
print(f"Total speakers: {len(set(e.get('child_id', '') for e in metadata))}")

In [ ]:
# ══════════════════════════════════════════════════════════════
# 7c. Train/Val Split Visualization by Age Bucket
# ══════════════════════════════════════════════════════════════

train_meta, val_meta = create_train_val_split(
    metadata,
    val_ratio=CONFIG["validation"]["split_ratio"],
    split_by=CONFIG["validation"]["split_by"],
    stratify_by=CONFIG["validation"]["stratify_by"],
)

train_bucket_counts = defaultdict(int)
val_bucket_counts = defaultdict(int)
for e in train_meta:
    train_bucket_counts[e.get("age_bucket", "unknown")] += 1
for e in val_meta:
    val_bucket_counts[e.get("age_bucket", "unknown")] += 1

all_buckets = sorted(set(list(train_bucket_counts.keys()) + list(val_bucket_counts.keys())))
train_counts = [train_bucket_counts[b] for b in all_buckets]
val_counts = [val_bucket_counts[b] for b in all_buckets]

x = np.arange(len(all_buckets))
width = 0.35

fig, ax = plt.subplots(figsize=(12, 5))
bars1 = ax.bar(x - width / 2, train_counts, width, label="Train", color="steelblue")
bars2 = ax.bar(x + width / 2, val_counts, width, label="Validation", color="coral")
ax.set_xlabel("Age Bucket")
ax.set_ylabel("Utterance Count")
ax.set_title("Train/Val Split by Age Bucket")
ax.set_xticks(x)
ax.set_xticklabels(all_buckets)
ax.legend()

for bar in bars1:
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 2,
            str(int(bar.get_height())), ha="center", va="bottom", fontsize=8)
for bar in bars2:
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 2,
            str(int(bar.get_height())), ha="center", va="bottom", fontsize=8)

plt.tight_layout()
plt.savefig("eda_train_val_split.png", dpi=150, bbox_inches="tight")
plt.show()

train_speakers = set(e.get("child_id") for e in train_meta)
val_speakers = set(e.get("child_id") for e in val_meta)
overlap = train_speakers & val_speakers
print(f"Train: {len(train_meta)} utterances, {len(train_speakers)} speakers")
print(f"Val:   {len(val_meta)} utterances, {len(val_speakers)} speakers")
print(f"Speaker overlap: {len(overlap)} (should be 0)")

In [ ]:
# ══════════════════════════════════════════════════════════════
# 7d. Audio Waveform + Mel Spectrogram Samples (4 random samples)
# ══════════════════════════════════════════════════════════════

rng = random.Random(42)
audio_dir_path = Path(AUDIO_DIR)
# Filter to entries with existing audio files
available_entries = [e for e in metadata if (audio_dir_path / e["audio_path"]).exists()]
sample_entries = rng.sample(available_entries, min(4, len(available_entries)))

fig, axes = plt.subplots(4, 2, figsize=(16, 16))
for i, entry in enumerate(sample_entries):
    audio_path = audio_dir_path / entry["audio_path"]
    audio, sr = load_audio(str(audio_path))
    duration = len(audio) / sr

    # Waveform
    t = np.linspace(0, duration, len(audio))
    axes[i, 0].plot(t, audio, color="steelblue", linewidth=0.5)
    axes[i, 0].set_xlim(0, duration)
    axes[i, 0].set_ylabel("Amplitude")
    label = entry.get("orthographic_text", "")[:50]
    age = entry.get("age_bucket", "?")
    axes[i, 0].set_title(f"Waveform | age={age} | \"{label}\" | {duration:.2f}s")
    if i == 3:
        axes[i, 0].set_xlabel("Time (s)")

    # Mel spectrogram
    S = librosa.feature.melspectrogram(y=audio, sr=sr, n_mels=80, fmax=8000)
    S_dB = librosa.power_to_db(S, ref=np.max)
    img = axes[i, 1].imshow(S_dB, aspect="auto", origin="lower",
                            extent=[0, duration, 0, 80], cmap="magma")
    axes[i, 1].set_ylabel("Mel bin")
    axes[i, 1].set_title(f"Mel Spectrogram | {entry.get('child_id', '?')}")
    if i == 3:
        axes[i, 1].set_xlabel("Time (s)")
    plt.colorbar(img, ax=axes[i, 1], format="%+2.0f dB")

plt.tight_layout()
plt.savefig("eda_audio_samples.png", dpi=150, bbox_inches="tight")
plt.show()
print("Audio samples visualized.")

---
## 8. AutoWhisper — Autonomous Experiment Loop

Inspired by [Karpathy's autoresearch](https://github.com/karpathy/autoresearch).
Runs time-boxed (15 min) hyperparameter experiments, keeps improvements, reverts regressions.

**Two modes:**
- **Scripted**: Pre-defined modification sequence (no API key needed)
- **Agent**: Claude API proposes modifications (requires `ANTHROPIC_API_KEY` secret)

> Set `RUN_AUTOWHISPER = True` in the cell below to enable.

In [ ]:
# ══════════════════════════════════════════════════════════════
# AutoWhisper Configuration
# ══════════════════════════════════════════════════════════════

RUN_AUTOWHISPER = False  # Set True to run experiments
AUTOWHISPER_MODE = "scripted"  # "scripted" or "agent"
AUTOWHISPER_TAG = "run_01"  # Change per session
AUTOWHISPER_MAX_EXPERIMENTS = 10
AUTOWHISPER_SESSION_BUFFER_MIN = 30
AUTOWHISPER_HF_REPO = "nishantgaurav23/autowhisper-results"
AUTOWHISPER_TIME_BUDGET = 900  # 15 min per experiment
AUTOWHISPER_EVAL_SAMPLES = 200

if RUN_AUTOWHISPER:
    AUTOWHISPER_SESSION_START = time.time()
    AUTOWHISPER_RESULTS_DIR = f"results/autowhisper/{AUTOWHISPER_TAG}"
    os.makedirs(AUTOWHISPER_RESULTS_DIR, exist_ok=True)
    AUTOWHISPER_LOG_PATH = f"{AUTOWHISPER_RESULTS_DIR}/results.tsv"
    print(f"AutoWhisper | Mode: {AUTOWHISPER_MODE} | Tag: {AUTOWHISPER_TAG}")
else:
    print("Skipping AutoWhisper (RUN_AUTOWHISPER=False)")

In [ ]:
# ═══ AutoWhisper — Evaluation Harness (IMMUTABLE) ═══

import hashlib
import csv

def aw_load_fast_eval_set(n_samples=200):
    """Load fixed, deterministic subset of validation data (shorter utterances first)."""
    all_meta = load_metadata(str(METADATA_PATH))
    _, val_meta = create_train_val_split(
        all_meta, val_ratio=0.1, split_by="child_id", stratify_by="age_bucket", seed=42)
    val_meta.sort(key=lambda e: (
        e.get("audio_duration_sec", 999),
        hashlib.md5(e.get("child_id", "").encode()).hexdigest()))
    return val_meta[:min(n_samples, len(val_meta))]

def aw_evaluate_wer(predictions, references):
    """Compute WER with normalizer. Returns dict with wer, S/I/D, n_samples."""
    norm_preds = [normalize_text(p) for p in predictions]
    norm_refs = [normalize_text(r) for r in references]
    pairs = [(p, r) for p, r in zip(norm_preds, norm_refs) if r.strip()]
    if not pairs:
        return {"wer": -1.0, "substitutions": 0, "deletions": 0, "insertions": 0, "n_samples": 0}
    fp, fr = zip(*pairs)
    output = jiwer.process_words(list(fr), list(fp))
    return {"wer": output.wer, "substitutions": output.substitutions,
            "deletions": output.deletions, "insertions": output.insertions, "n_samples": len(fr)}

def aw_get_peak_vram_mb():
    if torch.cuda.is_available():
        return int(torch.cuda.max_memory_allocated() / 1024 / 1024)
    return -1

def aw_init_log(path):
    with open(path, "w") as f:
        f.write("experiment_id\tcommit_hash\tval_wer\tpeak_vram_mb\tduration_sec\tstatus\tdescription\n")

def aw_append_log(path, row):
    with open(path, "a") as f:
        f.write("\t".join(str(row.get(k, "")) for k in
                ["experiment_id", "commit_hash", "val_wer", "peak_vram_mb",
                 "duration_sec", "status", "description"]) + "\n")

def aw_get_best_wer(path):
    best = float("inf")
    with open(path) as f:
        for row in csv.DictReader(f, delimiter="\t"):
            if row["status"] in ("baseline", "keep"):
                w = float(row["val_wer"])
                if 0 <= w < best:
                    best = w
    return best if best < float("inf") else 1.0

def aw_load_results(path):
    with open(path) as f:
        return list(csv.DictReader(f, delimiter="\t"))

print("AutoWhisper harness & logging loaded.")

In [ ]:
# ═══ AutoWhisper — Experiment Runner ═══

def aw_run_experiment(config, eval_set):
    """Run one experiment: load model, evaluate on fast eval set, return metrics."""
    t0 = time.time()
    try:
        processor = WhisperProcessor.from_pretrained(config["model_name"])
        model = WhisperForConditionalGeneration.from_pretrained(config["model_name"])

        if config.get("gradient_checkpointing", True):
            model.config.use_cache = False
            model.gradient_checkpointing_enable()
        model.config.apply_spec_augment = True
        model.config.mask_time_prob = config.get("spec_augment_mask_time_prob", 0.05)
        model.config.mask_feature_prob = config.get("spec_augment_mask_feature_prob", 0.04)
        model.config.forced_decoder_ids = None
        model.config.suppress_tokens = []

        if config.get("mode") == "lora":
            from peft import LoraConfig as _LC, get_peft_model as _gpm
            _lc = _LC(r=config.get("lora_r", 32), lora_alpha=config.get("lora_alpha", 64),
                      target_modules=config.get("lora_target_modules", ["q_proj", "v_proj"]),
                      lora_dropout=config.get("lora_dropout", 0.05), bias="none",
                      task_type="SEQ_2_SEQ_LM")
            model = _gpm(model, _lc)

        device = "cuda" if torch.cuda.is_available() else "cpu"
        model = model.to(device).eval()

        predictions, references = [], []
        with torch.no_grad():
            for item in eval_set:
                ap = AUDIO_DIR / item["audio_path"]
                if not Path(ap).exists():
                    continue
                audio, sr = load_audio(ap)
                inputs = processor(audio, sampling_rate=sr, return_tensors="pt").input_features.to(device)
                if device == "cuda":
                    inputs = inputs.half()
                ids = model.generate(inputs, num_beams=config.get("num_beams", 5),
                                     max_new_tokens=config.get("max_new_tokens", 225))
                predictions.append(processor.batch_decode(ids, skip_special_tokens=True)[0])
                references.append(item.get("orthographic_text", ""))
                if time.time() - t0 > AUTOWHISPER_TIME_BUDGET:
                    break

        result = aw_evaluate_wer(predictions, references)
        del model, processor
        if torch.cuda.is_available(): torch.cuda.empty_cache()
        return {"val_wer": result["wer"], "peak_vram_mb": aw_get_peak_vram_mb(),
                "duration_sec": int(time.time() - t0), "status": "ok"}
    except Exception as e:
        if torch.cuda.is_available(): torch.cuda.empty_cache()
        return {"val_wer": -1.0, "peak_vram_mb": aw_get_peak_vram_mb(),
                "duration_sec": int(time.time() - t0), "status": f"crash: {e}"}

print("Experiment runner loaded.")

In [ ]:
# ═══ AutoWhisper — Scripted Experiments Definition ═══

AW_BASELINE_CONFIG = {
    "model_name": "openai/whisper-small", "mode": "full",
    "learning_rate": 3e-5, "warmup_steps": 300, "max_steps": 500,
    "per_device_batch_size": 4, "gradient_accumulation": 8,
    "fp16": True, "gradient_checkpointing": True,
    "num_beams": 5, "max_new_tokens": 225,
    "spec_augment_mask_time_prob": 0.05, "spec_augment_mask_feature_prob": 0.04,
    "lora_r": 32, "lora_alpha": 64,
    "lora_target_modules": ["q_proj", "v_proj"], "lora_dropout": 0.05,
}

SCRIPTED_EXPERIMENTS = [
    {"description": "Lower learning rate to 1e-5", "patch": {"learning_rate": 1e-5}},
    {"description": "Increase warmup steps to 500", "patch": {"warmup_steps": 500}},
    {"description": "Higher learning rate 5e-5", "patch": {"learning_rate": 5e-5}},
    {"description": "Reduce gradient accumulation to 4", "patch": {"gradient_accumulation": 4}},
    {"description": "Increase beam width to 8", "patch": {"num_beams": 8}},
    {"description": "SpecAugment mask_time_prob 0.08", "patch": {"spec_augment_mask_time_prob": 0.08}},
    {"description": "SpecAugment mask_time_prob 0.02", "patch": {"spec_augment_mask_time_prob": 0.02}},
    {"description": "Increase max_steps to 750", "patch": {"max_steps": 750}},
    {"description": "Batch size 2 with grad_accum 16", "patch": {"per_device_batch_size": 2, "gradient_accumulation": 16}},
    {"description": "Reduce beam width to 3", "patch": {"num_beams": 3}},
]

print(f"{len(SCRIPTED_EXPERIMENTS)} scripted experiments defined.")

In [ ]:
# ═══ AutoWhisper — Run Experiment Loop ═══

if RUN_AUTOWHISPER:
    eval_set = aw_load_fast_eval_set(n_samples=AUTOWHISPER_EVAL_SAMPLES)
    print(f"Fast eval set: {len(eval_set)} samples")
    aw_init_log(AUTOWHISPER_LOG_PATH)

    # Baseline
    print("\nRunning baseline...")
    bl = aw_run_experiment(AW_BASELINE_CONFIG, eval_set)
    aw_append_log(AUTOWHISPER_LOG_PATH, {
        "experiment_id": "000", "commit_hash": "baseline",
        "val_wer": f"{bl['val_wer']:.4f}", "peak_vram_mb": bl["peak_vram_mb"],
        "duration_sec": bl["duration_sec"], "status": "baseline",
        "description": "Baseline configuration"})
    print(f"Baseline WER: {bl['val_wer']:.4f} | VRAM: {bl['peak_vram_mb']}MB | {bl['duration_sec']}s")

    # Scripted loop
    if AUTOWHISPER_MODE == "scripted":
        for i, exp in enumerate(SCRIPTED_EXPERIMENTS[:AUTOWHISPER_MAX_EXPERIMENTS]):
            exp_id = f"{i+1:03d}"
            print(f"\n{'='*60}")
            print(f"Experiment {exp_id}: {exp['description']}")
            print(f"{'='*60}")

            exp_config = {**AW_BASELINE_CONFIG, **exp["patch"]}
            result = aw_run_experiment(exp_config, eval_set)
            best = aw_get_best_wer(AUTOWHISPER_LOG_PATH)

            if result["val_wer"] >= 0 and result["val_wer"] < best:
                status = "keep"
                AW_BASELINE_CONFIG.update(exp["patch"])
                print(f"  -> KEEP (WER: {result['val_wer']:.4f}, was {best:.4f})")
            elif result["val_wer"] < 0:
                status = "crash"
                print(f"  -> CRASH")
            else:
                status = "discard"
                print(f"  -> DISCARD (WER: {result['val_wer']:.4f}, best: {best:.4f})")

            aw_append_log(AUTOWHISPER_LOG_PATH, {
                "experiment_id": exp_id, "commit_hash": status,
                "val_wer": f"{result['val_wer']:.4f}", "peak_vram_mb": result["peak_vram_mb"],
                "duration_sec": result["duration_sec"], "status": status,
                "description": exp["description"]})

    print("\nExperiment loop complete.")
else:
    print("Skipping AutoWhisper experiments.")

In [ ]:
# ═══ AutoWhisper — Results & Progress Plot ═══

if RUN_AUTOWHISPER:
    results = aw_load_results(AUTOWHISPER_LOG_PATH)

    print("\n" + "="*70)
    print("AutoWhisper Results Summary")
    print("="*70)
    print(f"{'ID':<5} {'Status':<10} {'WER':>8} {'VRAM':>8} {'Time':>7}  Description")
    print("-"*70)
    for r in results:
        print(f"{r['experiment_id']:<5} {r['status']:<10} {r['val_wer']:>8} "
              f"{r['peak_vram_mb']:>8} {r['duration_sec']:>7}  {r['description']}")

    # Progress scatter plot
    fig, ax = plt.subplots(figsize=(12, 5))
    colors = {"baseline": "blue", "keep": "green", "discard": "orange", "crash": "red"}
    for r in results:
        wer = float(r["val_wer"])
        if wer < 0: continue
        idx = int(r["experiment_id"])
        ax.scatter(idx, wer, c=colors.get(r["status"], "gray"), s=100,
                   zorder=5, edgecolors="black", linewidths=0.5)
        ax.annotate(r["experiment_id"], (idx, wer), textcoords="offset points",
                    xytext=(5, 5), fontsize=8)

    # Frontier line
    best, frontier_x, frontier_y = float("inf"), [], []
    for r in results:
        wer = float(r["val_wer"])
        if wer >= 0 and r["status"] in ("baseline", "keep"):
            if wer < best: best = wer
            frontier_x.append(int(r["experiment_id"]))
            frontier_y.append(best)
    if frontier_x:
        ax.plot(frontier_x, frontier_y, "g--", alpha=0.5, label="Best WER frontier")

    ax.set_xlabel("Experiment")
    ax.set_ylabel("Validation WER")
    ax.set_title(f"AutoWhisper Progress — {AUTOWHISPER_TAG}")
    ax.legend()
    plt.tight_layout()
    plt.savefig(f"{AUTOWHISPER_RESULTS_DIR}/autowhisper_progress.png", dpi=150, bbox_inches="tight")
    plt.show()
    print(f"\nBest WER: {aw_get_best_wer(AUTOWHISPER_LOG_PATH):.4f}")
else:
    print("No AutoWhisper results to display.")

In [ ]:
# ═══ AutoWhisper — Upload Results to HuggingFace Hub ═══

if RUN_AUTOWHISPER and PUSH_TO_HUB and os.environ.get("HF_TOKEN"):
    from huggingface_hub import HfApi
    _api = HfApi()
    try:
        _api.upload_file(path_or_fileobj=AUTOWHISPER_LOG_PATH,
                         path_in_repo=f"{AUTOWHISPER_TAG}/results.tsv",
                         repo_id=AUTOWHISPER_HF_REPO, repo_type="model")
        _plot = f"{AUTOWHISPER_RESULTS_DIR}/autowhisper_progress.png"
        if os.path.exists(_plot):
            _api.upload_file(path_or_fileobj=_plot,
                             path_in_repo=f"{AUTOWHISPER_TAG}/progress.png",
                             repo_id=AUTOWHISPER_HF_REPO, repo_type="model")
        print(f"Results uploaded to {AUTOWHISPER_HF_REPO}/{AUTOWHISPER_TAG}/")
    except Exception as e:
        print(f"Upload failed: {e}")
else:
    print("Skipping AutoWhisper upload.")

---
## 9. Augmented Retraining

Retrain both models with noise augmentation (RealClass + MUSAN).
Guarded by `RUN_AUGMENTED` flag.

In [ ]:
# ══════════════════════════════════════════════════════════════
# 8a. Verify Noise Directories
# ══════════════════════════════════════════════════════════════

if RUN_AUGMENTED:
    print("Checking noise directories for augmented training...")

    noise_ok = False
    realclass_ok = False

    if NOISE_DIR and Path(NOISE_DIR).exists():
        wav_files = list(Path(NOISE_DIR).glob("*.wav"))
        flac_files = list(Path(NOISE_DIR).glob("*.flac"))
        total_noise = len(wav_files) + len(flac_files)
        print(f"NOISE_DIR: {NOISE_DIR}")
        print(f"  .wav files: {len(wav_files)}, .flac files: {len(flac_files)}")
        noise_ok = total_noise > 0
    else:
        print(f"NOISE_DIR not found: {NOISE_DIR}")

    if MUSAN_DIR and Path(MUSAN_DIR).exists():
        wav_files = list(Path(MUSAN_DIR).rglob("*.wav"))
        flac_files = list(Path(MUSAN_DIR).rglob("*.flac"))
        total_musan = len(wav_files) + len(flac_files)
        print(f"MUSAN_DIR: {MUSAN_DIR}")
        print(f"  .wav files: {len(wav_files)}, .flac files: {len(flac_files)}")
        realclass_ok = total_musan > 0
    else:
        print(f"MUSAN_DIR not found: {MUSAN_DIR}")

    if noise_ok and realclass_ok:
        print("Both noise directories verified. Augmented training can proceed.")
    else:
        print("WARNING: Missing noise directories. Augmented training will be skipped.")
        RUN_AUGMENTED = False
else:
    print("Skipping augmented training (RUN_AUGMENTED=False)")

In [ ]:
# ══════════════════════════════════════════════════════════════
# 8b. Whisper-small Augmented Retraining
# ══════════════════════════════════════════════════════════════

if RUN_AUGMENTED:
    print("=== Whisper-small Augmented Retraining ===")

    aug_cfg = CONFIG["augmentation"]
    augment_fn = create_augmentation(
        noise_dir=str(MUSAN_DIR),
        realclass_dir=str(NOISE_DIR),
        sample_rate=CONFIG["sample_rate"],
        realclass_min_snr=aug_cfg["realclass_min_snr_db"],
        realclass_max_snr=aug_cfg["realclass_max_snr_db"],
        musan_min_snr=aug_cfg["musan_min_snr_db"],
        musan_max_snr=aug_cfg["musan_max_snr_db"],
    )

    SMALL_AUG_OUTPUT_DIR = Path(str(SMALL_OUTPUT_DIR).replace("whisper-small", "whisper-small-augmented"))
    SMALL_AUG_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

    # Build augmented datasets
    small_aug_train_ds, small_aug_val_ds = build_datasets(
        CONFIG["whisper_small"], str(METADATA_PATH), str(AUDIO_DIR),
        augment_fn=augment_fn,
    )
    print(f"Augmented Train: {len(small_aug_train_ds)}, Val: {len(small_aug_val_ds)}")

    # Load model
    small_aug_model = WhisperForConditionalGeneration.from_pretrained(
        CONFIG["whisper_small"]["model_name"], torch_dtype=torch.float32
    )
    small_aug_processor = WhisperProcessor.from_pretrained(CONFIG["whisper_small"]["model_name"])

    if CONFIG["whisper_small"].get("gradient_checkpointing", True):
        small_aug_model.gradient_checkpointing_enable()

    sa = CONFIG["spec_augment"]
    if sa["apply"]:
        small_aug_model.config.apply_spec_augment = True
        small_aug_model.config.mask_time_prob = sa["mask_time_prob"]
        small_aug_model.config.mask_time_length = sa["mask_time_length"]
        small_aug_model.config.mask_feature_prob = sa["mask_feature_prob"]
        small_aug_model.config.mask_feature_length = sa["mask_feature_length"]

    small_aug_model.config.forced_decoder_ids = None
    small_aug_model.generation_config.forced_decoder_ids = None
    small_aug_model.generation_config.suppress_tokens = []

    # Build augmented config with different hub ID
    small_aug_config = dict(CONFIG["whisper_small"])
    small_aug_config["hub_model_id"] = "nishantgaurav23/pasketti-whisper-small-augmented"

    training_args = make_training_args(small_aug_config, SMALL_AUG_OUTPUT_DIR, push_to_hub=PUSH_TO_HUB)
    collator = WhisperDataCollator(pad_token_id=small_aug_processor.tokenizer.pad_token_id)
    compute_metrics_fn = make_compute_metrics(small_aug_processor.tokenizer)

    trainer = Seq2SeqTrainer(
        model=small_aug_model,
        args=training_args,
        train_dataset=small_aug_train_ds,
        eval_dataset=small_aug_val_ds,
        data_collator=collator,
        compute_metrics=compute_metrics_fn,
        processing_class=small_aug_processor.feature_extractor,
    )

    print("Starting augmented Whisper-small training...")
    t0 = time.time()
    trainer.train()
    elapsed = time.time() - t0
    print(f"Augmented training completed in {elapsed/3600:.1f} hours")

    metrics = trainer.evaluate()
    print(f"Augmented Validation WER: {metrics.get('eval_wer', -1):.4f}")

    trainer.save_model()
    small_aug_processor.save_pretrained(str(SMALL_AUG_OUTPUT_DIR))
    print(f"Augmented model saved to {SMALL_AUG_OUTPUT_DIR}")

    del small_aug_model, trainer
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
else:
    print("Skipping Whisper-small augmented retraining (RUN_AUGMENTED=False)")

In [ ]:
# ══════════════════════════════════════════════════════════════
# 8c. Whisper-large-v3 LoRA Augmented Retraining
# ══════════════════════════════════════════════════════════════

if RUN_AUGMENTED:
    from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

    print("=== Whisper-large-v3 LoRA Augmented Retraining ===")

    aug_cfg = CONFIG["augmentation"]
    augment_fn = create_augmentation(
        noise_dir=str(MUSAN_DIR),
        realclass_dir=str(NOISE_DIR),
        sample_rate=CONFIG["sample_rate"],
        realclass_min_snr=aug_cfg["realclass_min_snr_db"],
        realclass_max_snr=aug_cfg["realclass_max_snr_db"],
        musan_min_snr=aug_cfg["musan_min_snr_db"],
        musan_max_snr=aug_cfg["musan_max_snr_db"],
    )

    LORA_AUG_OUTPUT_DIR = Path(str(LORA_OUTPUT_DIR).replace("whisper-large-v3-lora", "whisper-large-v3-lora-augmented"))
    LORA_AUG_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

    lora_aug_train_ds, lora_aug_val_ds = build_datasets(
        CONFIG["whisper_large_v3"], str(METADATA_PATH), str(AUDIO_DIR),
        augment_fn=augment_fn,
    )
    print(f"Augmented Train: {len(lora_aug_train_ds)}, Val: {len(lora_aug_val_ds)}")

    lora_aug_cfg = CONFIG["whisper_large_v3"]
    use_8bit = lora_aug_cfg.get("load_in_8bit", False)
    load_kwargs = {}
    if use_8bit:
        load_kwargs["load_in_8bit"] = True
        load_kwargs["device_map"] = "auto"
    else:
        load_kwargs["torch_dtype"] = torch.float16 if lora_aug_cfg.get("fp16", True) else torch.float32

    lora_aug_model = WhisperForConditionalGeneration.from_pretrained(
        lora_aug_cfg["model_name"], **load_kwargs
    )
    lora_aug_processor = WhisperProcessor.from_pretrained(lora_aug_cfg["model_name"])

    if use_8bit:
        lora_aug_model = prepare_model_for_kbit_training(lora_aug_model)
    if lora_aug_cfg.get("gradient_checkpointing", True):
        lora_aug_model.gradient_checkpointing_enable()

    sa = CONFIG["spec_augment"]
    if sa["apply"]:
        lora_aug_model.config.apply_spec_augment = True
        lora_aug_model.config.mask_time_prob = sa["mask_time_prob"]
        lora_aug_model.config.mask_time_length = sa["mask_time_length"]
        lora_aug_model.config.mask_feature_prob = sa["mask_feature_prob"]
        lora_aug_model.config.mask_feature_length = sa["mask_feature_length"]

    lora_aug_model.config.forced_decoder_ids = None
    lora_aug_model.config.suppress_tokens = []
    lora_aug_model.generation_config.forced_decoder_ids = None

    lp = lora_aug_cfg["lora"]
    peft_config = LoraConfig(
        r=lp["r"], lora_alpha=lp["alpha"], target_modules=lp["target_modules"],
        lora_dropout=lp["dropout"], bias=lp["bias"], task_type=lp["task_type"],
    )
    lora_aug_model = get_peft_model(lora_aug_model, peft_config)
    lora_aug_model.print_trainable_parameters()

    lora_aug_hub_config = dict(CONFIG["whisper_large_v3"])
    lora_aug_hub_config["hub_model_id"] = "nishantgaurav23/pasketti-whisper-lora-augmented"

    training_args = make_training_args(lora_aug_hub_config, LORA_AUG_OUTPUT_DIR, push_to_hub=PUSH_TO_HUB)
    collator = WhisperDataCollator(pad_token_id=lora_aug_processor.tokenizer.pad_token_id)
    compute_metrics_fn = make_compute_metrics(lora_aug_processor.tokenizer)

    trainer = Seq2SeqTrainer(
        model=lora_aug_model,
        args=training_args,
        train_dataset=lora_aug_train_ds,
        eval_dataset=lora_aug_val_ds,
        data_collator=collator,
        compute_metrics=compute_metrics_fn,
        processing_class=lora_aug_processor.feature_extractor,
    )

    print("Starting augmented LoRA training...")
    t0 = time.time()
    trainer.train()
    elapsed = time.time() - t0
    print(f"Augmented LoRA training completed in {elapsed/3600:.1f} hours")

    metrics = trainer.evaluate()
    print(f"Augmented LoRA Validation WER: {metrics.get('eval_wer', -1):.4f}")

    lora_aug_model.save_pretrained(str(LORA_AUG_OUTPUT_DIR))
    lora_aug_processor.save_pretrained(str(LORA_AUG_OUTPUT_DIR))
    print(f"Augmented LoRA adapter saved to {LORA_AUG_OUTPUT_DIR}")

    del lora_aug_model, trainer
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
else:
    print("Skipping LoRA augmented retraining (RUN_AUGMENTED=False)")

---
## 10. Evaluation & Error Analysis

Load a trained model, run inference on validation set, compute WER breakdowns,
detect hallucinations, compare clean vs noisy, and visualize training curves.

In [ ]:
# ══════════════════════════════════════════════════════════════
# 9a. Load Trained Model & Run Inference on Validation Set
# ══════════════════════════════════════════════════════════════

if RUN_EVALUATION:
    from transformers import pipeline as hf_pipeline

    # Try to load from checkpoint dirs, fall back to base model
    eval_model_path = None
    eval_model_label = None
    for candidate, label in [
        (SMALL_OUTPUT_DIR, "Whisper-small (fine-tuned)"),
        (LORA_OUTPUT_DIR, "Whisper-large-v3-LoRA"),
    ]:
        if Path(candidate).exists() and (Path(candidate) / "config.json").exists():
            eval_model_path = str(candidate)
            eval_model_label = label
            break

    if eval_model_path is None:
        eval_model_path = "openai/whisper-small"
        eval_model_label = "Whisper-small (zero-shot)"
        print(f"No trained checkpoint found, using base model: {eval_model_path}")
    else:
        print(f"Evaluating: {eval_model_label} from {eval_model_path}")

    device_id = 0 if torch.cuda.is_available() else -1
    pipe = hf_pipeline(
        "automatic-speech-recognition",
        model=eval_model_path,
        device=device_id,
        chunk_length_s=30,
    )

    # Get validation split
    eval_metadata = load_metadata(str(METADATA_PATH))
    _, eval_val_meta = create_train_val_split(
        eval_metadata,
        val_ratio=CONFIG["validation"]["split_ratio"],
        split_by=CONFIG["validation"]["split_by"],
        stratify_by=CONFIG["validation"]["stratify_by"],
    )
    print(f"Validation set: {len(eval_val_meta)} utterances")

    # Run inference
    eval_refs, eval_hyps, eval_ages, eval_ids = [], [], [], []
    eval_audio_dir = Path(AUDIO_DIR)
    t0 = time.time()
    for i, entry in enumerate(eval_val_meta):
        audio_path = eval_audio_dir / entry["audio_path"]
        if not audio_path.exists():
            continue
        try:
            result = pipe(str(audio_path))
            hyp = normalize_and_postprocess(result["text"])
        except Exception as exc:
            hyp = ""
            print(f"  Error on {entry.get('utterance_id', i)}: {exc}")

        ref = normalize_text(entry.get("orthographic_text", ""))
        eval_refs.append(ref)
        eval_hyps.append(hyp)
        eval_ages.append(entry.get("age_bucket", "unknown"))
        eval_ids.append(entry.get("utterance_id", str(i)))

        if (i + 1) % 200 == 0:
            print(f"  Processed {i + 1}/{len(eval_val_meta)}...")

    eval_elapsed = time.time() - t0
    print(f"Inference done in {eval_elapsed:.1f}s ({len(eval_refs)} samples)")
else:
    print("Skipping evaluation (RUN_EVALUATION=False)")

In [ ]:
# ══════════════════════════════════════════════════════════════
# 9b. Overall WER + Per-Age WER Breakdown
# ══════════════════════════════════════════════════════════════

if RUN_EVALUATION and eval_refs:
    overall_wer = compute_wer(eval_refs, eval_hyps)
    per_age = compute_per_age_wer(eval_refs, eval_hyps, eval_ages)

    print(f"\n{'='*50}")
    print(f"Model: {eval_model_label}")
    print(f"Overall WER: {overall_wer:.4f} ({overall_wer*100:.2f}%)")
    print(f"{'='*50}")
    print(f"{'Age Bucket':<15} {'WER':>8} {'Count':>8}")
    print(f"{'-'*35}")

    age_wer_data = {}
    for age in sorted(per_age.keys()):
        count = sum(1 for a in eval_ages if a == age)
        print(f"{age:<15} {per_age[age]:>8.4f} {count:>8}")
        age_wer_data[age] = {"wer": per_age[age], "count": count}

    # Bar chart
    fig, ax = plt.subplots(figsize=(10, 5))
    ages_sorted = sorted(age_wer_data.keys())
    wer_vals = [age_wer_data[a]["wer"] for a in ages_sorted]
    count_vals = [age_wer_data[a]["count"] for a in ages_sorted]
    bars = ax.bar(ages_sorted, wer_vals, color="steelblue", edgecolor="white")
    ax.axhline(y=overall_wer, color="red", linestyle="--", label=f"Overall WER: {overall_wer:.4f}")
    ax.axhline(y=0.20, color="green", linestyle=":", alpha=0.7, label="Target: 0.20")
    ax.set_xlabel("Age Bucket")
    ax.set_ylabel("WER")
    ax.set_title(f"WER by Age Bucket ({eval_model_label})")
    ax.legend()
    for bar, w, c in zip(bars, wer_vals, count_vals):
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.005,
                f"{w:.3f}\n(n={c})", ha="center", va="bottom", fontsize=8)
    plt.tight_layout()
    plt.savefig("eval_wer_by_age.png", dpi=150, bbox_inches="tight")
    plt.show()

In [ ]:
# ══════════════════════════════════════════════════════════════
# 9c. S/I/D Error Breakdown
# ══════════════════════════════════════════════════════════════

if RUN_EVALUATION and eval_refs:
    total_S, total_I, total_D, total_H, total_ref_words = 0, 0, 0, 0, 0

    for ref, hyp in zip(eval_refs, eval_hyps):
        if not ref.strip():
            continue
        try:
            output = jiwer.process_words(ref, hyp)
            total_S += output.substitutions
            total_I += output.insertions
            total_D += output.deletions
            total_H += output.hits
            total_ref_words += output.substitutions + output.deletions + output.hits
        except Exception:
            continue

    total_errors = total_S + total_I + total_D
    print(f"\nError Breakdown (total reference words: {total_ref_words}):")
    print(f"  Substitutions: {total_S:>6} ({total_S/max(total_ref_words,1)*100:.1f}%)")
    print(f"  Insertions:    {total_I:>6} ({total_I/max(total_ref_words,1)*100:.1f}%)")
    print(f"  Deletions:     {total_D:>6} ({total_D/max(total_ref_words,1)*100:.1f}%)")
    print(f"  Hits:          {total_H:>6} ({total_H/max(total_ref_words,1)*100:.1f}%)")
    print(f"  Total errors:  {total_errors:>6}")

    # Pie chart
    fig, ax = plt.subplots(figsize=(7, 7))
    labels = ["Substitutions", "Insertions", "Deletions", "Correct (Hits)"]
    sizes = [total_S, total_I, total_D, total_H]
    colors_pie = ["#e74c3c", "#f39c12", "#3498db", "#2ecc71"]
    explode = (0.05, 0.05, 0.05, 0)
    ax.pie(sizes, explode=explode, labels=labels, colors=colors_pie, autopct="%1.1f%%",
           shadow=False, startangle=140)
    ax.set_title("Error Type Distribution (S/I/D/H)")
    plt.savefig("eval_error_breakdown.png", dpi=150, bbox_inches="tight")
    plt.show()

In [ ]:
# ══════════════════════════════════════════════════════════════
# 9d. Hallucination Detection
# ══════════════════════════════════════════════════════════════

if RUN_EVALUATION and eval_refs:
    hallucinations = []
    for uid, ref, hyp in zip(eval_ids, eval_refs, eval_hyps):
        ref_words = len(ref.split()) if ref.strip() else 0
        hyp_words = len(hyp.split()) if hyp.strip() else 0
        # Flag: hypothesis has >3x more words than reference
        if ref_words > 0 and hyp_words > 3 * ref_words:
            hallucinations.append({
                "utterance_id": uid,
                "ref": ref,
                "hyp": hyp,
                "ref_words": ref_words,
                "hyp_words": hyp_words,
                "ratio": hyp_words / ref_words,
            })
        # Also flag: ref is empty but hyp is long
        elif ref_words == 0 and hyp_words > 5:
            hallucinations.append({
                "utterance_id": uid,
                "ref": ref,
                "hyp": hyp,
                "ref_words": ref_words,
                "hyp_words": hyp_words,
                "ratio": float("inf"),
            })

    print(f"\nHallucination Detection (hyp_words > 3x ref_words):")
    print(f"  Total flagged: {len(hallucinations)} / {len(eval_refs)} "
          f"({len(hallucinations)/max(len(eval_refs),1)*100:.1f}%)")

    if hallucinations:
        print(f"\n  Top 10 worst hallucinations:")
        for h in sorted(hallucinations, key=lambda x: x["hyp_words"], reverse=True)[:10]:
            print(f"    [{h['utterance_id']}] ref({h['ref_words']}w): \"{h['ref'][:40]}\"")
            print(f"      hyp({h['hyp_words']}w): \"{h['hyp'][:60]}\"")
    else:
        print("  No hallucinations detected.")

In [ ]:
# ══════════════════════════════════════════════════════════════
# 9e. Clean vs Noisy WER Comparison
# ══════════════════════════════════════════════════════════════

if RUN_EVALUATION and eval_refs and NOISE_DIR and Path(NOISE_DIR).exists():
    print("Running noisy WER comparison...")

    # Create noise augmentation for validation
    noisy_augment = create_augmentation(
        noise_dir=str(MUSAN_DIR) if MUSAN_DIR and Path(MUSAN_DIR).exists() else str(NOISE_DIR),
        realclass_dir=str(NOISE_DIR),
        sample_rate=CONFIG["sample_rate"],
    )

    noisy_hyps = []
    for entry in eval_val_meta:
        audio_path = eval_audio_dir / entry["audio_path"]
        if not audio_path.exists():
            continue
        try:
            audio, sr = load_audio(str(audio_path))
            noisy_audio = noisy_augment(audio, sample_rate=sr)
            # Save to temp file for pipeline
            import soundfile as sf
            tmp_path = Path(tempfile.mktemp(suffix=".wav"))
            sf.write(str(tmp_path), noisy_audio, sr)
            result = pipe(str(tmp_path))
            noisy_hyps.append(normalize_and_postprocess(result["text"]))
            tmp_path.unlink(missing_ok=True)
        except Exception:
            noisy_hyps.append("")

    if len(noisy_hyps) == len(eval_refs):
        clean_wer = compute_wer(eval_refs, eval_hyps)
        noisy_wer = compute_wer(eval_refs, noisy_hyps)

        print(f"\n  Clean WER: {clean_wer:.4f}")
        print(f"  Noisy WER: {noisy_wer:.4f}")
        print(f"  Degradation: {(noisy_wer - clean_wer)*100:.2f} percentage points")

        fig, ax = plt.subplots(figsize=(8, 5))
        bars = ax.bar(["Clean", "Noisy"], [clean_wer, noisy_wer],
                      color=["steelblue", "coral"], edgecolor="white", width=0.5)
        ax.set_ylabel("WER")
        ax.set_title("Clean vs Noisy Validation WER")
        ax.axhline(y=0.20, color="green", linestyle=":", alpha=0.7, label="Target: 0.20")
        ax.legend()
        for bar in bars:
            ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.005,
                    f"{bar.get_height():.4f}", ha="center", va="bottom", fontsize=11)
        plt.tight_layout()
        plt.savefig("eval_clean_vs_noisy.png", dpi=150, bbox_inches="tight")
        plt.show()
    else:
        print(f"  Mismatch: {len(noisy_hyps)} noisy vs {len(eval_refs)} clean refs")
elif RUN_EVALUATION:
    print("Skipping clean vs noisy comparison (no noise directory available)")

In [ ]:
# ══════════════════════════════════════════════════════════════
# 9f. Training Curves Visualization
# ══════════════════════════════════════════════════════════════

if RUN_EVALUATION:
    def plot_training_curves(checkpoint_dir, model_label):
        """Read trainer_state.json and plot loss + WER curves."""
        state_path = None
        cp_dir = Path(checkpoint_dir)
        # Check main dir and checkpoint subdirs
        for candidate in [cp_dir / "trainer_state.json"] + sorted(cp_dir.glob("checkpoint-*/trainer_state.json")):
            if candidate.exists():
                state_path = candidate
                break

        if state_path is None:
            print(f"  No trainer_state.json found in {checkpoint_dir}")
            return False

        with open(state_path) as f:
            state = json.load(f)

        log_history = state.get("log_history", [])
        if not log_history:
            print(f"  Empty log_history in {state_path}")
            return False

        train_steps, train_losses = [], []
        eval_steps, eval_losses, eval_wers = [], [], []

        for entry in log_history:
            step = entry.get("step", 0)
            if "loss" in entry and "eval_loss" not in entry:
                train_steps.append(step)
                train_losses.append(entry["loss"])
            if "eval_loss" in entry:
                eval_steps.append(step)
                eval_losses.append(entry["eval_loss"])
                if "eval_wer" in entry:
                    eval_wers.append(entry["eval_wer"])

        fig, axes = plt.subplots(1, 2, figsize=(14, 5))

        # Loss curves
        if train_steps:
            axes[0].plot(train_steps, train_losses, label="Train Loss", alpha=0.7)
        if eval_steps:
            axes[0].plot(eval_steps, eval_losses, label="Eval Loss", marker="o", markersize=4)
        axes[0].set_xlabel("Step")
        axes[0].set_ylabel("Loss")
        axes[0].set_title(f"{model_label} — Training & Eval Loss")
        axes[0].legend()

        # WER curve
        if eval_wers and eval_steps:
            axes[1].plot(eval_steps[:len(eval_wers)], eval_wers, marker="o", color="coral", markersize=4)
            axes[1].axhline(y=0.20, color="green", linestyle=":", alpha=0.7, label="Target: 0.20")
            axes[1].set_xlabel("Step")
            axes[1].set_ylabel("WER")
            axes[1].set_title(f"{model_label} — Validation WER")
            axes[1].legend()
        else:
            axes[1].text(0.5, 0.5, "No WER data", ha="center", va="center", transform=axes[1].transAxes)
            axes[1].set_title(f"{model_label} — Validation WER (no data)")

        plt.tight_layout()
        safe_label = model_label.lower().replace(" ", "_").replace("-", "_")
        plt.savefig(f"eval_training_curves_{safe_label}.png", dpi=150, bbox_inches="tight")
        plt.show()
        return True

    found_any = False
    for ckpt_dir, label in [
        (SMALL_OUTPUT_DIR, "Whisper-small"),
        (LORA_OUTPUT_DIR, "Whisper-large-v3 LoRA"),
    ]:
        if Path(ckpt_dir).exists():
            print(f"\nPlotting training curves for {label}...")
            if plot_training_curves(ckpt_dir, label):
                found_any = True

    if not found_any:
        print("No training state files found. Train a model first to see curves.")

In [ ]:
# ══════════════════════════════════════════════════════════════
# 9g. Model Comparison Bar Charts
# ══════════════════════════════════════════════════════════════

if RUN_EVALUATION:
    model_results = {}

    # Try to gather WER from each available checkpoint
    for ckpt_dir, label, model_cfg in [
        (SMALL_OUTPUT_DIR, "Small (FT)", CONFIG["whisper_small"]),
        (LORA_OUTPUT_DIR, "Large-v3 LoRA", CONFIG["whisper_large_v3"]),
    ]:
        state_path = None
        cp = Path(ckpt_dir)
        for candidate in [cp / "trainer_state.json"] + sorted(cp.glob("checkpoint-*/trainer_state.json")):
            if candidate.exists():
                state_path = candidate
                break
        if state_path:
            with open(state_path) as f:
                state = json.load(f)
            # Get best eval WER from log history
            best_wer = None
            for entry in state.get("log_history", []):
                if "eval_wer" in entry:
                    w = entry["eval_wer"]
                    if best_wer is None or w < best_wer:
                        best_wer = w
            if best_wer is not None:
                model_results[label] = best_wer

    # Add current evaluation result if available
    if eval_refs:
        model_results[eval_model_label.split("(")[0].strip()] = overall_wer

    if model_results:
        fig, ax = plt.subplots(figsize=(10, 5))
        names = list(model_results.keys())
        wers = list(model_results.values())
        colors_bar = plt.cm.Set2(np.linspace(0, 1, len(names)))
        bars = ax.bar(names, wers, color=colors_bar, edgecolor="white", width=0.5)
        ax.axhline(y=0.20, color="green", linestyle=":", alpha=0.7, label="Target: 0.20")
        ax.set_ylabel("WER")
        ax.set_title("Model Comparison — Best Validation WER")
        ax.legend()
        for bar, w in zip(bars, wers):
            ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.005,
                    f"{w:.4f}", ha="center", va="bottom", fontsize=11, fontweight="bold")
        plt.tight_layout()
        plt.savefig("eval_model_comparison.png", dpi=150, bbox_inches="tight")
        plt.show()
    else:
        print("No model results available for comparison chart.")

---
## 11. Ensemble Inference & Submission

Two-model ensemble: Whisper-large-v3 + LoRA (Model A) and Whisper-small fine-tuned (Model B).
Confidence-based merging: prefer Model A, fall back to Model B on empty predictions.

In [ ]:
# ══════════════════════════════════════════════════════════════
# 10a. Ensemble Helper Functions (inline from submission/main.py)
# ══════════════════════════════════════════════════════════════

SAMPLE_RATE_ENSEMBLE = 16000
DEFAULT_MODEL_ENSEMBLE = "openai/whisper-small"
LARGE_MODEL_ENSEMBLE = "openai/whisper-large-v3"


def get_device():
    """Return best available device: cuda > mps > cpu."""
    if torch.cuda.is_available():
        return "cuda"
    if hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
        return "mps"
    return "cpu"


def maybe_compile_model(model, device):
    """Optionally compile model with torch.compile on CUDA."""
    if device != "cuda":
        return model
    try:
        return torch.compile(model, mode="reduce-overhead")
    except Exception:
        logger.warning("torch.compile failed, using uncompiled model")
        return model


def get_beam_config(model_size="small"):
    """Return beam search configuration based on model size."""
    if model_size == "large":
        return {"num_beams": 8, "length_penalty": 1.0, "max_new_tokens": 225}
    return {"num_beams": 5, "length_penalty": 1.0, "max_new_tokens": 225}


def load_small_model(device, model_name_or_path=DEFAULT_MODEL_ENSEMBLE):
    """Load Whisper-small model + processor."""
    dtype = torch.float16 if device == "cuda" else torch.float32
    attn_kwargs = {}
    if device == "cuda":
        attn_kwargs["attn_implementation"] = "sdpa"
    model = WhisperForConditionalGeneration.from_pretrained(
        model_name_or_path, torch_dtype=dtype, **attn_kwargs
    ).to(device)
    model = maybe_compile_model(model, device)
    model.eval()
    try:
        processor = WhisperProcessor.from_pretrained(model_name_or_path)
    except OSError:
        processor = WhisperProcessor.from_pretrained(DEFAULT_MODEL_ENSEMBLE)
    return model, processor


def load_large_model_ensemble(device, base_model=LARGE_MODEL_ENSEMBLE, adapter_path=None):
    """Load Whisper-large-v3 base model + LoRA adapter."""
    from peft import PeftModel
    dtype = torch.float16 if device == "cuda" else torch.float32
    attn_kwargs = {}
    if device == "cuda":
        attn_kwargs["attn_implementation"] = "sdpa"
    base = WhisperForConditionalGeneration.from_pretrained(
        base_model, torch_dtype=dtype, **attn_kwargs
    ).to(device)
    model = PeftModel.from_pretrained(base, str(adapter_path))
    model = maybe_compile_model(model, device)
    model.eval()
    processor = WhisperProcessor.from_pretrained(base_model)
    return model, processor


def transcribe_batch_ensemble(model, processor, audio_arrays, device,
                              num_beams=5, length_penalty=1.0, max_new_tokens=225):
    """Transcribe a batch of audio arrays. Returns list of raw text predictions."""
    if not audio_arrays:
        return []
    inputs = processor(audio_arrays, sampling_rate=SAMPLE_RATE_ENSEMBLE,
                       return_tensors="pt", padding=True)
    input_features = inputs.input_features.to(device)
    if device == "cuda":
        input_features = input_features.to(torch.float16)
    with torch.no_grad():
        generated = model.generate(
            input_features, language="en", task="transcribe",
            num_beams=num_beams, max_new_tokens=max_new_tokens,
            length_penalty=length_penalty,
        )
    return processor.batch_decode(generated, skip_special_tokens=True)


def run_model_inference(model, processor, utterances, audio_dir, device,
                        batch_size=16, model_size="small"):
    """Run inference on all utterances. Returns {utterance_id: normalized_text}."""
    sorted_utts = sorted(utterances, key=lambda u: u.get("audio_duration_sec", 0), reverse=True)
    beam_config = get_beam_config(model_size)
    predictions = {}
    audio_base = Path(audio_dir)

    for i in range(0, len(sorted_utts), batch_size):
        batch = sorted_utts[i : i + batch_size]
        audio_arrays, batch_ids = [], []

        for u in batch:
            uid = u.get("utterance_id", u.get("audio_path", ""))
            audio_path = audio_base / u["audio_path"]
            try:
                audio, _ = load_audio(str(audio_path), target_sr=SAMPLE_RATE_ENSEMBLE)
            except Exception:
                predictions[uid] = ""
                continue
            if is_silence(audio):
                predictions[uid] = ""
                continue
            audio_arrays.append(audio)
            batch_ids.append(uid)

        if audio_arrays:
            texts = transcribe_batch_ensemble(model, processor, audio_arrays, device, **beam_config)
            for uid, text in zip(batch_ids, texts):
                predictions[uid] = normalize_and_postprocess(text)

    return predictions


def merge_predictions(preds_a, preds_b):
    """Merge: prefer Model A, fall back to Model B on empty."""
    if preds_b is None:
        return dict(preds_a)
    merged = {}
    all_ids = set(list(preds_a.keys()) + list(preds_b.keys()))
    for uid in all_ids:
        text_a = preds_a.get(uid, "")
        if text_a.strip():
            merged[uid] = text_a
        else:
            merged[uid] = preds_b.get(uid, "")
    return merged


def run_ensemble_inference(utterances, audio_dir, device, adapter_path=None,
                           small_model_path=DEFAULT_MODEL_ENSEMBLE, batch_size=16):
    """Run two-model ensemble inference."""
    t0 = time.time()
    preds_a, preds_b = None, None

    # Model A: large-v3 + LoRA
    if adapter_path and Path(adapter_path).exists():
        try:
            logger.info("Loading Model A (Whisper-large-v3 + LoRA)...")
            model_a, proc_a = load_large_model_ensemble(device, adapter_path=adapter_path)
            preds_a = run_model_inference(model_a, proc_a, utterances, audio_dir, device,
                                          batch_size, model_size="large")
            logger.info("Model A done in %.1fs", time.time() - t0)
            del model_a, proc_a
            if device == "cuda":
                torch.cuda.empty_cache()
        except Exception as exc:
            logger.warning("Model A failed: %s", exc)

    # Model B: small fine-tuned
    try:
        logger.info("Loading Model B (Whisper-small)...")
        model_b, proc_b = load_small_model(device, model_name_or_path=small_model_path)
        preds_b = run_model_inference(model_b, proc_b, utterances, audio_dir, device,
                                      batch_size, model_size="small")
        logger.info("Model B done in %.1fs", time.time() - t0)
        del model_b, proc_b
        if device == "cuda":
            torch.cuda.empty_cache()
    except Exception as exc:
        logger.warning("Model B failed: %s", exc)

    if preds_a is not None:
        return merge_predictions(preds_a, preds_b)
    if preds_b is not None:
        return preds_b
    return {u.get("utterance_id", ""): "" for u in utterances}


def write_submission(predictions, utterances, output_path):
    """Write submission.jsonl."""
    output_path = Path(output_path)
    output_path.parent.mkdir(parents=True, exist_ok=True)
    with open(output_path, "w") as f:
        for u in utterances:
            uid = u.get("utterance_id", u.get("audio_path", ""))
            line = {"utterance_id": uid, "orthographic_text": predictions.get(uid, "")}
            f.write(json.dumps(line) + "\n")
    return output_path


print("Ensemble helper functions defined.")

In [ ]:
# ══════════════════════════════════════════════════════════════
# 10b. Run Ensemble on Validation Data
# ══════════════════════════════════════════════════════════════

if RUN_ENSEMBLE:
    device = get_device()
    print(f"Device: {device}")

    ensemble_metadata = load_metadata(str(METADATA_PATH))
    _, ensemble_val_meta = create_train_val_split(
        ensemble_metadata,
        val_ratio=CONFIG["validation"]["split_ratio"],
        split_by=CONFIG["validation"]["split_by"],
        stratify_by=CONFIG["validation"]["stratify_by"],
    )
    print(f"Ensemble validation set: {len(ensemble_val_meta)} utterances")

    # Resolve model paths
    small_path = str(SMALL_OUTPUT_DIR) if (Path(SMALL_OUTPUT_DIR) / "config.json").exists() else DEFAULT_MODEL_ENSEMBLE
    lora_path = str(LORA_OUTPUT_DIR) if Path(LORA_OUTPUT_DIR).exists() else None

    print(f"Small model: {small_path}")
    print(f"LoRA adapter: {lora_path}")

    batch_size = 16 if device == "cuda" else 4

    t0 = time.time()
    ensemble_preds = run_ensemble_inference(
        utterances=ensemble_val_meta,
        audio_dir=str(AUDIO_DIR),
        device=device,
        adapter_path=lora_path,
        small_model_path=small_path,
        batch_size=batch_size,
    )
    ensemble_elapsed = time.time() - t0
    print(f"\nEnsemble inference done in {ensemble_elapsed:.1f}s")
    print(f"Predictions: {len(ensemble_preds)}")
else:
    print("Skipping ensemble inference (RUN_ENSEMBLE=False)")

In [ ]:
# ══════════════════════════════════════════════════════════════
# 10c. Show Sample Predictions with Match/Diff Indicators
# ══════════════════════════════════════════════════════════════

if RUN_ENSEMBLE and ensemble_preds:
    print(f"\n{'='*70}")
    print("Sample Predictions (first 20)")
    print(f"{'='*70}")

    sample_count = min(20, len(ensemble_val_meta))
    match_count = 0
    total_shown = 0

    for entry in ensemble_val_meta[:sample_count]:
        uid = entry.get("utterance_id", entry.get("audio_path", ""))
        ref = normalize_text(entry.get("orthographic_text", ""))
        hyp = ensemble_preds.get(uid, "")
        is_match = ref.strip().lower() == hyp.strip().lower()
        indicator = "MATCH" if is_match else "DIFF"
        if is_match:
            match_count += 1
        total_shown += 1

        print(f"\n[{indicator}] {uid}")
        print(f"  REF: {ref}")
        print(f"  HYP: {hyp}")

    print(f"\n{'='*70}")
    print(f"Exact match rate: {match_count}/{total_shown} ({match_count/max(total_shown,1)*100:.1f}%)")

    # Compute ensemble WER on full validation set
    ens_refs, ens_hyps = [], []
    for entry in ensemble_val_meta:
        uid = entry.get("utterance_id", entry.get("audio_path", ""))
        ref = normalize_text(entry.get("orthographic_text", ""))
        hyp = ensemble_preds.get(uid, "")
        if ref.strip():
            ens_refs.append(ref)
            ens_hyps.append(hyp)

    if ens_refs:
        ens_wer = compute_wer(ens_refs, ens_hyps)
        print(f"Ensemble WER on validation: {ens_wer:.4f} ({ens_wer*100:.2f}%)")

In [ ]:
# ══════════════════════════════════════════════════════════════
# 10d. Write Submission File
# ══════════════════════════════════════════════════════════════

if RUN_ENSEMBLE and ensemble_preds:
    submission_path = Path("submission.jsonl")
    write_submission(ensemble_preds, ensemble_val_meta, submission_path)
    print(f"Submission written to: {submission_path}")
    print(f"Total predictions: {len(ensemble_preds)}")

    # Show first 5 lines
    with open(submission_path) as f:
        lines = f.readlines()
    print(f"\nFirst 5 lines of submission.jsonl:")
    for line in lines[:5]:
        print(f"  {line.strip()}")

In [ ]:
# ══════════════════════════════════════════════════════════════
# 10e. Validate Submission Format
# ══════════════════════════════════════════════════════════════

if RUN_ENSEMBLE and ensemble_preds:
    submission_path = Path("submission.jsonl")
    if submission_path.exists():
        with open(submission_path) as f:
            sub_lines = f.readlines()

        errors = []
        seen_ids = set()
        for i, line in enumerate(sub_lines):
            try:
                entry = json.loads(line.strip())
            except json.JSONDecodeError:
                errors.append(f"Line {i+1}: Invalid JSON")
                continue

            if "utterance_id" not in entry:
                errors.append(f"Line {i+1}: Missing 'utterance_id'")
            if "orthographic_text" not in entry:
                errors.append(f"Line {i+1}: Missing 'orthographic_text'")

            uid = entry.get("utterance_id", "")
            if uid in seen_ids:
                errors.append(f"Line {i+1}: Duplicate utterance_id '{uid}'")
            seen_ids.add(uid)

            text = entry.get("orthographic_text", "")
            if not isinstance(text, str):
                errors.append(f"Line {i+1}: 'orthographic_text' is not a string")

        expected_ids = set(
            e.get("utterance_id", e.get("audio_path", "")) for e in ensemble_val_meta
        )
        missing_ids = expected_ids - seen_ids
        extra_ids = seen_ids - expected_ids

        print(f"Submission Validation:")
        print(f"  Total lines:     {len(sub_lines)}")
        print(f"  Unique IDs:      {len(seen_ids)}")
        print(f"  Expected IDs:    {len(expected_ids)}")
        print(f"  Missing IDs:     {len(missing_ids)}")
        print(f"  Extra IDs:       {len(extra_ids)}")
        print(f"  Format errors:   {len(errors)}")

        if errors:
            print(f"\n  Errors:")
            for e in errors[:10]:
                print(f"    {e}")
        else:
            print("\n  Submission format is VALID.")
    else:
        print("No submission file found.")

---
## 12. Summary

All stages of the ChildWhisper pipeline: data exploration, training, augmentation,
evaluation, and ensemble inference.

In [ ]:
# ══════════════════════════════════════════════════════════════
# 11. Summary Table
# ══════════════════════════════════════════════════════════════

print("\n" + "=" * 70)
print("ChildWhisper Training & Evaluation Summary")
print("=" * 70)

summary_rows = []

# Data stats
summary_rows.append(("Dataset", f"{len(all_entries)} utterances", ""))

# Training stages
for ckpt_dir, label in [
    (SMALL_OUTPUT_DIR, "Whisper-small FT"),
    (LORA_OUTPUT_DIR, "Large-v3 LoRA"),
]:
    status = "Not run"
    wer_str = "-"
    cp = Path(ckpt_dir)
    if cp.exists():
        state_path = None
        for candidate in [cp / "trainer_state.json"] + sorted(cp.glob("checkpoint-*/trainer_state.json")):
            if candidate.exists():
                state_path = candidate
                break
        if state_path:
            with open(state_path) as f:
                state = json.load(f)
            best_wer = None
            total_steps = 0
            for entry in state.get("log_history", []):
                if "eval_wer" in entry:
                    w = entry["eval_wer"]
                    if best_wer is None or w < best_wer:
                        best_wer = w
                total_steps = max(total_steps, entry.get("step", 0))
            if best_wer is not None:
                wer_str = f"{best_wer:.4f}"
            status = f"Done ({total_steps} steps)"
        elif (cp / "config.json").exists():
            status = "Checkpoint saved"
    summary_rows.append((label, status, wer_str))

# Augmented
if RUN_AUGMENTED:
    summary_rows.append(("Small Augmented", "Done", "-"))
    summary_rows.append(("LoRA Augmented", "Done", "-"))
else:
    summary_rows.append(("Augmented Training", "Skipped", "-"))

# Evaluation
if RUN_EVALUATION and eval_refs:
    summary_rows.append((f"Eval ({eval_model_label})", f"{len(eval_refs)} samples", f"{overall_wer:.4f}"))
else:
    summary_rows.append(("Evaluation", "Skipped", "-"))

# Ensemble
if RUN_ENSEMBLE and ensemble_preds:
    ens_wer_str = f"{ens_wer:.4f}" if ens_refs else "-"
    summary_rows.append(("Ensemble", f"{len(ensemble_preds)} predictions", ens_wer_str))
else:
    summary_rows.append(("Ensemble", "Skipped", "-"))

print(f"\n{'Stage':<25} {'Status':<30} {'Best WER':>10}")
print(f"{'-'*65}")
for stage, status, wer in summary_rows:
    print(f"{stage:<25} {status:<30} {wer:>10}")

print(f"\n{'='*70}")
print("Outputs:")
if Path(SMALL_OUTPUT_DIR).exists():
    print(f"  Whisper-small checkpoint: {SMALL_OUTPUT_DIR}")
if Path(LORA_OUTPUT_DIR).exists():
    print(f"  LoRA adapter checkpoint:  {LORA_OUTPUT_DIR}")
if Path("submission.jsonl").exists():
    print(f"  Submission file:          submission.jsonl")
print("\nNext steps:")
print("  1. Download checkpoints and upload to HuggingFace Hub")
print("  2. Package submission for competition")
print("  3. Review error analysis and tune augmentation parameters")
print("=" * 70)